<a href="https://colab.research.google.com/github/rahul02500/Practicepython/blob/main/Build%20a%20Multi%20user%20conversatione%20tool%20calling%20agentic%20ai%20research%20assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from warnings import filterwarnings
filterwarnings('ignore')

In [7]:
!pip install langchain==1.3.11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.1/671.1 kB 24.2 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.2.11
    Uninstalling langsmith-0.2.11:
      Successfully uninstalled langsmith-0.2.11
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.63
    Uninstalling langchain-core-0.3.63:
      Successfully uninstalled langchain-core-0.3.63
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.25
    Uninstalling langchain-0.3.25:
      Successfully uninstalled langchain-0.3.25
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=

In [8]:
!pip install markitdown


In [9]:
!pip install langchain-openai==0.3.0


  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
Using cached langchain_core-0.3.86-py3-none-any.whl (461 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.8
    Uninstalling langchain-core-1.4.8:
      Successfully uninstalled langchain-core-1.4.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.3.11 requires langchain-core<2.0.0,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langchain-community 0.3.14 requires langchain<0.4.0,>=0.3.14, but you have langchain 1.3.11 which is incompatible.
langchain-community 0.3.14 requires langsmith<0.3,>=0.1.125, but you have langsmith 0.9.7 which is incompatible.
langgraph 1.2.6 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, bu

In [10]:
from google.colab import userdata

# Replace 'OPENAI_API_KEY' with the exact name you used in Colab Secrets
Openai_key = userdata.get('OpenAI')

In [11]:
from google.colab import userdata

TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')

In [12]:
from google.colab import userdata

Weatherapi = userdata.get('Weatherapi')

In [13]:
!pip install langchain-community==0.3.14

  Using cached langchain-0.3.30-py3-none-any.whl.metadata (6.4 kB)
  Using cached langsmith-0.2.11-py3-none-any.whl.metadata (14 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_core-0.3.85-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain-0.3.29-py3-none-any.whl.metadata (6.4 kB)
  Using cached langchain-0.3.28-py3-none-any.whl.metadata (6.4 kB)
  Using cached langchain_core-0.3.84-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.83-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is still looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_core-0.3.82-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.81-py3-none-any.whl.m

In [14]:
import os
os.environ['OPENAI']=Openai_key
os.environ['TAVILY_API_KEY']=TAVILY_API_KEY

In [18]:
from langchain_core.tools import tool
from markitdown import MarkItDown
from langchain_community.tools.tavily_search import TavilySearchResults
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, TimeoutError
import requests
import json
from warnings import filterwarnings
filterwarnings('ignore')

tavily_tool = TavilySearchResults(max_results=5,
                                  search_depth='advanced',
                                  include_answer=False,
                                  include_raw_content=True)
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br"
})
md = MarkItDown(requests_session=session)

@tool
def search_web_extract_info(query: str) -> list:
    """Search the web for a query and extracts useful information from the search links."""
    print('Calling web search tool')
    results = tavily_tool.invoke(query)
    docs = []

    def extract_content(url):
        """Helper function to extract content from a URL."""
        extracted_info = md.convert(url)
        text_title = extracted_info.title.strip()
        text_content = extracted_info.text_content.strip()
        return text_title + '\n' + text_content

    with ThreadPoolExecutor() as executor:
        for result in tqdm(results):
            try:
                future = executor.submit(extract_content, result['url'])
                # Wait for up to 60 seconds for the task to complete
                content = future.result(timeout=60)
                docs.append(content)
            except TimeoutError:
                print(f"Extraction timed out for url: {result['url']}")
            except Exception as e:
                print(f"Error extracting from url: {result['url']} - {e}")

    return docs


@tool
def get_weather(query: str) -> list:
    """Search weatherapi to get the current weather of the queried location."""
    print('Calling weather tool')
    base_url = "http://api.weatherapi.com/v1/current.json"
    complete_url = f"{base_url}?key={Weatherapi}&q={query}"

    response = requests.get(complete_url)
    data = response.json()
    if data.get("location"):
        return data
    else:
        return "Weather Data Not Found"

In [22]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model="gpt-4o",temperature=0, api_key=Openai_key)
tools = [search_web_extract_info,get_weather]

chatgpt_with_tools = chatgpt.bind_tools(tools)

In [24]:
propmt = "Get details of Microsoft's earnings call Q4 2024"
response = chatgpt_with_tools.invoke(propmt)
response.tool_calls

[{'name': 'search_web_extract_info',
  'args': {'query': 'Microsoft earnings call Q4 2024 details'},
  'id': 'call_khSthbuPV3PEQqXBEZkYSSCT',
  'type': 'tool_call'}]

In [25]:
propmt = "how is the weather in Bangalore today"
response = chatgpt_with_tools.invoke(propmt)
response.tool_calls

[{'name': 'get_weather',
  'args': {'query': 'Bangalore'},
  'id': 'call_oEqK9OgqXNVRGFeTTPeExBvn',
  'type': 'tool_call'}]

In [26]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYS_PROMPT = """Act as a helpful assistant.
                You run in a loop of Thought, Action, PAUSE, Observation.
                At the end of the loop, you output an Answer.
                Use Thought to describe your thoughts about the question you have been asked.
                Use Action to run one of the actions available to you - then return PAUSE.
                Observation will be the result of running those actions.
                Repeat till you get to the answer for the given user query.

                Use the following workflow format:
                  Question: the input task you must solve
                  Thought: you should always think about what to do
                  Action: the action to take which can be any of the following:
                            - break it into smaller steps if needed
                            - see if you can answer the given task with your trained knowledge
                            - call the most relevant tools at your disposal mentioned below in case you need more information
                  Action Input: the input to the action
                  Observation: the result of the action
                  ... (this Thought/Action/Action Input/Observation can repeat N times)
                  Thought: I now know the final answer
                  Final Answer: the final answer to the original input question

                Tools at your disposal to perform tasks as needed:
                  - get_weather: whenever user asks get the weather of a place.
                  - search_web_extract_info: whenever user asks for specific information or if you don't know the answer.
             """

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", SYS_PROMPT),
        MessagesPlaceholder(variable_name="history", optional=True),
        ("human", "{query}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

prompt_template.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="Act as a helpful assistant.\n                You run in a loop of Thought, Action, PAUSE, Observation.\n                At the end of the loop, you output an Answer.\n                Use Thought to describe your thoughts about the question you have been asked.\n                Use Action to run one of the actions available to you - then return PAUSE.\n                Observation will be the result of running those actions.\n                Repeat till you get to the answer for the given user query.\n\n                Use the following workflow format:\n                  Question: the input task you must solve\n                  Thought: you should always think about what to do\n                  Action: the action to take which can be any of the following:\n                            - break it into smaller steps if needed\n                            - see if you can

In [28]:
from langchain.agents import create_tool_calling_agent
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model="gpt-4o",temperature=0, api_key=Openai_key)
tools = [search_web_extract_info, get_weather]
agent = create_tool_calling_agent(chatgpt, tools, prompt_template)

In [29]:
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent,
                               tools=tools,
                               early_stopping_method='force',
                               max_iterations=10)

In [30]:
query = """Summarize the key points discussed in Nvidia's Q4 2024 earnings call"""
response = chatgpt.invoke(query)
response.content

"As of my last update, Nvidia's Q4 2024 earnings call has not occurred, as my training data only goes up to October 2023. Therefore, I don't have access to the specific details of that earnings call. However, I can provide a general idea of what typically might be discussed in such a call:\n\n1. **Financial Performance**: Discussion of revenue, net income, and earnings per share compared to previous quarters and the same quarter in the previous year.\n\n2. **Segment Performance**: Insights into how different business segments, such as gaming, data center, professional visualization, and automotive, have performed.\n\n3. **Market Trends**: Commentary on market conditions affecting Nvidia's business, such as demand for GPUs, AI, and data center products.\n\n4. **Product Updates**: Announcements or updates on new products or technologies that Nvidia has released or plans to release.\n\n5. **Strategic Initiatives**: Information on strategic partnerships, acquisitions, or other initiatives 

In [38]:
query = """Summarize the key points discussed in Nvidia's Q4 2024 earnings call"""
response = agent_executor.invoke ({"query": query})

Calling web search tool


 20%|██        | 1/5 [00:00<00:01,  3.89it/s]

Error extracting from url: https://seekingalpha.com/article/4672199-nvidia-corporation-nvda-q4-2024-earnings-call-transcript - 403 Client Error: Forbidden for url: https://seekingalpha.com/article/4672199-nvidia-corporation-nvda-q4-2024-earnings-call-transcript


 60%|██████    | 3/5 [00:01<00:01,  1.86it/s]

Error extracting from url: https://investor.nvidia.com/news/press-release-details/2024/NVIDIA-Announces-Financial-Results-for-Fourth-Quarter-and-Fiscal-2024 - 403 Client Error: Forbidden for url: https://investor.nvidia.com/news/press-release-details/2024/NVIDIA-Announces-Financial-Results-for-Fourth-Quarter-and-Fiscal-2024


100%|██████████| 5/5 [00:02<00:00,  1.83it/s]


RateLimitError: Error code: 429 - {'error': {'message': 'Request too large for gpt-4o in organization org-m23k87gg7RxP1eqpehjuKDOG on tokens per min (TPM): Limit 30000, Requested 30139. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [32]:
response

AIMessage(content="As of my last update, Nvidia's Q4 2024 earnings call has not occurred, as my training data only goes up to October 2023. Therefore, I don't have access to the specific details of that earnings call. However, I can provide a general idea of what typically might be discussed in such a call:\n\n1. **Financial Performance**: Discussion of revenue, net income, and earnings per share compared to previous quarters and the same quarter in the previous year.\n\n2. **Segment Performance**: Insights into how different business segments, such as gaming, data center, professional visualization, and automotive, have performed.\n\n3. **Market Trends**: Commentary on market conditions affecting Nvidia's business, such as demand for GPUs, AI, and data center products.\n\n4. **Product Updates**: Announcements or updates on new products or technologies that Nvidia has released or plans to release.\n\n5. **Strategic Initiatives**: Information on strategic partnerships, acquisitions, or 

In [40]:
from IPython.display import display, Markdown

display(Markdown(response.content))

As of my last update, Nvidia's Q4 2024 earnings call has not occurred, as my training data only goes up to October 2023. Therefore, I don't have access to the specific details of that earnings call. However, I can provide a general idea of what typically might be discussed in such a call:

1. **Financial Performance**: Discussion of revenue, net income, and earnings per share compared to previous quarters and the same quarter in the previous year.

2. **Segment Performance**: Insights into how different business segments, such as gaming, data center, professional visualization, and automotive, have performed.

3. **Market Trends**: Commentary on market conditions affecting Nvidia's business, such as demand for GPUs, AI, and data center products.

4. **Product Updates**: Announcements or updates on new products or technologies that Nvidia has released or plans to release.

5. **Strategic Initiatives**: Information on strategic partnerships, acquisitions, or other initiatives aimed at driving future growth.

6. **Guidance**: Forward-looking statements regarding expectations for the next quarter or fiscal year, including revenue and margin forecasts.

7. **Q&A Session**: Responses to questions from analysts and investors, providing additional insights into Nvidia's business strategy and market outlook.

For the most accurate and up-to-date information, you would need to refer to the official transcript or summary of Nvidia's Q4 2024 earnings call once it becomes available.

In [41]:
query = """Summarize the key points discussed in Intel's Q4 2024 earnings call"""
response = agent_executor.invoke({"query": query})
display(Markdown(response['output']))

Calling web search tool


 40%|████      | 2/5 [00:00<00:00,  3.28it/s]

Error extracting from url: https://download.intel.com/newsroom/2025/c8e6h3a2/intel-q4-2024y-earnings.pdf - File conversion failed after 1 attempts:
 - PdfConverter threw MissingDependencyException with message: PdfConverter recognized the input as a potential .pdf file, but the dependencies needed to read .pdf files have not been installed. To resolve this error, include the optional dependency [pdf] or [all] when installing MarkItDown. For example:

* pip install markitdown[pdf]
* pip install markitdown[all]
* pip install markitdown[pdf, ...]
* etc.



 80%|████████  | 4/5 [00:01<00:00,  2.81it/s]

Error extracting from url: https://www.sec.gov/Archives/edgar/data/50863/000005086324000147/q324_earningsrelease.htm - 403 Client Error: Forbidden for url: https://www.sec.gov/Archives/edgar/data/50863/000005086324000147/q324_earningsrelease.htm


100%|██████████| 5/5 [00:02<00:00,  2.14it/s]


RateLimitError: Error code: 429 - {'error': {'message': 'Request too large for gpt-4o in organization org-m23k87gg7RxP1eqpehjuKDOG on tokens per min (TPM): Limit 30000, Requested 51135. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [42]:
query = """which company's future outlook looks to be better?
        """
response = agent_executor.invoke({"query": query})
display(Markdown(response['output']))

To provide an accurate assessment of a company's future outlook, I would need to know which specific companies you are interested in comparing. Please provide the names of the companies you would like to evaluate.

In [43]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# used to retrieve conversation history from database
# based on a specific user or session ID
def get_session_history_db(session_id):
    return SQLChatMessageHistory(session_id, "sqlite:///memory.db")

# create a conversation chain + agent which can load memory based on specific user or session id
agentic_chatbot = RunnableWithMessageHistory(
    agent_executor,
    get_session_history_db,
    input_messages_key="query",
    history_messages_key="history",
)

# function to call the agent show results per user session
from IPython.display import display, Markdown
def chat_with_agent(prompt: str, session_id: str):
    response = agentic_chatbot.invoke({"query": prompt},
                                      {'configurable': { 'session_id': session_id}})
    display(Markdown(response['output']))

In [44]:
user_id = 'john001'
prompt = "Summarize the key points discussed in Nvidia's Q4 2024 earnings call"
chat_with_agent(prompt, user_id)

/usr/local/lib/python3.12/dist-packages/langchain_core/runnables/history.py:596: LangChainDeprecationWarning: `connection_string` was deprecated in LangChain 0.2.2 and will be removed in 1.0. Use connection instead.
  message_history = self.get_session_history(


Calling web search tool


 20%|██        | 1/5 [00:00<00:01,  3.76it/s]

Error extracting from url: https://seekingalpha.com/article/4672199-nvidia-corporation-nvda-q4-2024-earnings-call-transcript - 403 Client Error: Forbidden for url: https://seekingalpha.com/article/4672199-nvidia-corporation-nvda-q4-2024-earnings-call-transcript


 60%|██████    | 3/5 [00:01<00:01,  1.95it/s]

Error extracting from url: https://investor.nvidia.com/news/press-release-details/2024/NVIDIA-Announces-Financial-Results-for-Fourth-Quarter-and-Fiscal-2024 - 403 Client Error: Forbidden for url: https://investor.nvidia.com/news/press-release-details/2024/NVIDIA-Announces-Financial-Results-for-Fourth-Quarter-and-Fiscal-2024


100%|██████████| 5/5 [00:02<00:00,  1.99it/s]


RateLimitError: Error code: 429 - {'error': {'message': 'Request too large for gpt-4o in organization org-m23k87gg7RxP1eqpehjuKDOG on tokens per min (TPM): Limit 30000, Requested 30139. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [45]:
prompt = "What about Intel?"
chat_with_agent(prompt, user_id)

Calling web search tool


 60%|██████    | 3/5 [00:02<00:01,  1.78it/s]

Error extracting from url: https://finance.yahoo.com/quote/INTC/press-releases - 404 Client Error: Not Found for url: https://finance.yahoo.com/quote/INTC/press-releases/


100%|██████████| 5/5 [00:02<00:00,  2.00it/s]

Error extracting from url: https://finance.yahoo.com/quote/INTC/news - 404 Client Error: Not Found for url: https://finance.yahoo.com/quote/INTC/news/


Intel has been actively involved in various developments and announcements recently. Here are some highlights:

1. **Financial Updates**: Intel is set to report its second-quarter 2026 financial results after the market closes on July 23, 2026.

2. **Product Innovations**: Intel and MSI have co-engineered the world’s first Arc G3 handheld, focusing on high-performance handheld gaming.

3. **Corporate Initiatives**: Intel is advancing U.S. innovation, AI, and manufacturing as part of its America 250 initiative, highlighting its commitment to technological progress and national development.

4. **Leadership Changes**: Intel has announced a leadership appointment at Intel Foundry to accelerate development and manufacturing.

5. **Corporate Responsibility**: Intel's CEO, Lip-Bu Tan, has outlined progress in rebuilding execution, strengthening core businesses, and advancing responsible growth in the AI era in the 2025-26 Corporate Responsibility Report.

6. **Technological Advancements**: Intel has introduced new AI innovations at Computex, including chip-to-rackscale AI solutions delivered with the help of strategic industry partners.

These updates reflect Intel's ongoing efforts in innovation, leadership, and corporate responsibility. If you have a specific area of interest regarding Intel, please let me know!

In [46]:
prompt = "Which company seems to be doing better?"
chat_with_agent(prompt, user_id)

To determine which company is doing better, we would need to compare Intel with another specific company. This comparison could be based on various factors such as financial performance, market share, innovation, leadership, and other metrics.

If you have a specific company in mind that you would like to compare with Intel, please let me know, and I can help gather relevant information to make a comparison.

In [47]:
user_id = 'bond007'
prompt = "how is the weather in Bangalore today? Show detailed statistics"
chat_with_agent(prompt, user_id)

Calling weather tool


The weather in Bangalore today is partly cloudy. Here are the detailed statistics:

- **Temperature**: 21.2°C (70.2°F)
- **Condition**: Partly cloudy
- **Wind**: 18.1 mph (29.2 kph) from the west-southwest (WSW) at 245°
- **Pressure**: 1011.0 mb (29.85 in)
- **Precipitation**: 0.0 mm (0.0 in)
- **Humidity**: 78%
- **Cloud Cover**: 50%
- **Feels Like**: 21.2°C (70.2°F)
- **Wind Chill**: 20.2°C (68.4°F)
- **Dew Point**: 16.5°C (61.6°F)
- **Visibility**: 6.0 km (3.0 miles)
- **UV Index**: 0.0
- **Gusts**: Up to 25.3 mph (40.6 kph)
- **Chance of Rain**: 13%
- **Chance of Snow**: 0%

The local time in Bangalore is 01:34 AM.

In [48]:
user_id = 'bond007'
prompt = "what about Dubai?"
chat_with_agent(prompt, user_id)

Calling weather tool


The weather in Dubai today is clear. Here are the detailed statistics:

- **Temperature**: 35.1°C (95.2°F)
- **Condition**: Clear
- **Wind**: 2.2 mph (3.6 kph) from the south-southeast (SSE) at 153°
- **Pressure**: 998.0 mb (29.47 in)
- **Precipitation**: 0.0 mm (0.0 in)
- **Humidity**: 39%
- **Cloud Cover**: 0%
- **Feels Like**: 37.9°C (100.2°F)
- **Wind Chill**: 31.1°C (88.0°F)
- **Dew Point**: 24.5°C (76.1°F)
- **Visibility**: 10.0 km (6.0 miles)
- **UV Index**: 0.0
- **Gusts**: Up to 2.4 mph (3.9 kph)
- **Chance of Rain**: 5%
- **Chance of Snow**: 0%

The local time in Dubai is 12:04 AM.

In [49]:
user_id = 'bond007'
prompt = "which city is hotter?"
chat_with_agent(prompt, user_id)

Dubai is hotter than Bangalore today, with a temperature of 35.1°C (95.2°F) compared to Bangalore's 21.2°C (70.2°F).

### Resolving `RateLimitError: 429`

The `RateLimitError: 429 Too Many Requests` status code indicates that the user has sent too many requests in a given amount of time. This is a common mechanism used by APIs to prevent abuse and ensure fair usage.

Here are the primary ways to handle and resolve a `RateLimitError: 429`:

1.  **Introduce Delays (Throttling)**:
    *   The simplest solution is to pause between requests. If an API has a limit of, say, 100 requests per minute, you can calculate the appropriate delay (e.g., 0.6 seconds per request) and add `time.sleep()` calls in your loop.
    *   **Example (Python)**:
        ```python
        import time
        import requests

        for i in range(10):
            try:
                response = requests.get('https://api.example.com/data')
                response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
                print(f"Request {i+1} successful!")
            except requests.exceptions.HTTPError as e:
                if e.response.status_code == 429:
                    print(f"Rate limited on request {i+1}. Waiting...")
                    time.sleep(5) # Wait for 5 seconds before retrying
                    continue
                else:
                    raise
            time.sleep(1) # Wait 1 second between requests
        ```

2.  **Implement Retry Logic with Exponential Backoff**:
    *   This is a more robust approach. When you encounter a `429` error, you wait for a short period (e.g., 1 second) and retry. If it fails again, you double the waiting time (e.g., 2 seconds), then 4 seconds, and so on, up to a maximum number of retries or a maximum wait time.
    *   Many libraries (like `requests` with `requests-retry` or `tenacity`) provide decorators or built-in functionalities for exponential backoff.
    *   **Example (Conceptual)**:
        ```python
        from tenacity import retry, wait_exponential, stop_after_attempt
        import requests

        @retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
        def make_api_call(url):
            print("Attempting API call...")
            response = requests.get(url)
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            return response

        try:
            result = make_api_call('https://api.example.com/data')
            print("API call successful:", result.json())
        except requests.exceptions.RequestException as e:
            print("API call failed after multiple retries:", e)
        ```

3.  **Check `Retry-After` Header**:
    *   Some APIs include a `Retry-After` header in their `429` responses. This header tells you exactly how many seconds to wait before making another request.
    *   Always prioritize this header if it's provided.
    *   **Example (Python)**:
        ```python
        import time
        import requests

        response = requests.get('https://api.example.com/data')
        if response.status_code == 429:
            retry_after = response.headers.get('Retry-After')
            if retry_after:
                wait_time = int(retry_after)
                print(f"Rate limited. Retrying after {wait_time} seconds.")
                time.sleep(wait_time)
                # Then retry the request
            else:
                print("Rate limited, but no Retry-After header. Using default backoff.")
                time.sleep(5)
        ```

4.  **Increase Your API Rate Limit**:
    *   If you consistently hit rate limits and your application requires higher throughput, check the API provider's documentation. Many services offer higher rate limits for paid plans or enterprise users.

5.  **Batch Requests**:
    *   If the API supports it, try to combine multiple operations into a single batch request to reduce the total number of API calls.